In [1]:
import sys
sys.path.append("..")

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer

from src.data import make_splits
from src.lora import LoraESM, train_lora, predict_lora, count_trainable, trainable_state
from src.train import set_seed

device = "cuda"
MODEL = "facebook/esm2_t12_35M_UR50D"

train, val, test = make_splits("../data/deeploc_data.fasta")
print(len(train), len(val), len(test))               # 8867 2218 2773

le = LabelEncoder().fit(train.location)
y_train, y_val = le.transform(train.location), le.transform(val.location)
class_names = list(le.classes_)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
set_seed(0)
model = LoraESM(MODEL, n_classes=len(class_names))
print("trainable parameters:", count_trainable(model))     # expect 310,986

C:\dev\protein-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8867 2218 2773


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1514.24it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable parameters: 310986


In [2]:
SEEDS = [0]          # add 1 and 2 later if time allows

rows = []
for seed in SEEDS:
    print(f"===== seed {seed} =====")
    set_seed(seed)
    model = LoraESM(MODEL, n_classes=len(class_names))
    model, hist = train_lora(model, tokenizer, train.seq.tolist(), y_train,
                             val.seq.tolist(), y_val, device, seed)
    torch.save(trainable_state(model), f"../outputs/lora_esm2_seed{seed}.pt")
    pred = predict_lora(model, tokenizer, val.seq.tolist(), device)
    rows.append({"model": "esm2_35M_lora+mlp", "seed": seed,
                 "val_acc": accuracy_score(y_val, pred),
                 "val_macro_f1": f1_score(y_val, pred, average="macro")})
    print(rows[-1])

pd.DataFrame(rows).to_csv("../outputs/lora_esm2_results.csv", index=False)
print(classification_report(y_val, pred, target_names=class_names, digits=3, zero_division=0))

===== seed 0 =====


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 2344.32it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1  loss 1.0176  val acc 0.746  val macro-F1 0.545  (4.8 min)
epoch 2  loss 0.6543  val acc 0.770  val macro-F1 0.622  (4.7 min)
epoch 3  loss 0.5424  val acc 0.787  val macro-F1 0.646  (4.7 min)
epoch 4  loss 0.4662  val acc 0.790  val macro-F1 0.660  (4.7 min)
{'model': 'esm2_35M_lora+mlp', 'seed': 0, 'val_acc': 0.7899008115419297, 'val_macro_f1': 0.6604203588203517}
                       precision    recall  f1-score   support

        Cell.membrane      0.792     0.799     0.795       214
            Cytoplasm      0.677     0.655     0.666       406
Endoplasmic.reticulum      0.648     0.746     0.694       138
        Extracellular      0.901     0.924     0.912       316
      Golgi.apparatus      0.846     0.386     0.530        57
     Lysosome/Vacuole      0.250     0.077     0.118        52
        Mitochondrion      0.854     0.843     0.848       242
              Nucleus      0.815     0.892     0.852       647
           Peroxisome      0.462     0.240     0.316   

In [2]:
import gc
import os

gc.collect()
torch.cuda.empty_cache()

csv_path = "../outputs/lora_esm2_results.csv"
rows = pd.read_csv(csv_path).to_dict("records") if os.path.exists(csv_path) else []

for seed in [1, 2]:
    print(f"===== seed {seed} =====")
    set_seed(seed)
    model = LoraESM(MODEL, n_classes=len(class_names))
    model, hist = train_lora(model, tokenizer, train.seq.tolist(), y_train,
                             val.seq.tolist(), y_val, device, seed)
    torch.save(trainable_state(model), f"../outputs/lora_esm2_seed{seed}.pt")
    pred = predict_lora(model, tokenizer, val.seq.tolist(), device)
    rows.append({"model": "esm2_35M_lora+mlp", "seed": seed,
                 "val_acc": accuracy_score(y_val, pred),
                 "val_macro_f1": f1_score(y_val, pred, average="macro")})
    print(rows[-1])
    pd.DataFrame(rows).to_csv(csv_path, index=False)      # saved after every seed

res = pd.DataFrame(rows)
display(res.groupby("model")[["val_acc", "val_macro_f1"]].agg(["mean", "std"]).round(3))

===== seed 1 =====


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1274.97it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1  loss 1.0264  val acc 0.748  val macro-F1 0.553  (4.8 min)
epoch 2  loss 0.6603  val acc 0.779  val macro-F1 0.645  (4.8 min)
epoch 3  loss 0.5552  val acc 0.787  val macro-F1 0.660  (4.8 min)
epoch 4  loss 0.4668  val acc 0.787  val macro-F1 0.665  (4.8 min)
{'model': 'esm2_35M_lora+mlp', 'seed': 1, 'val_acc': 0.7867448151487827, 'val_macro_f1': 0.6645954460582019}
===== seed 2 =====


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1444.00it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1  loss 1.0099  val acc 0.758  val macro-F1 0.578  (4.8 min)
epoch 2  loss 0.6516  val acc 0.779  val macro-F1 0.630  (4.8 min)
epoch 3  loss 0.5465  val acc 0.788  val macro-F1 0.655  (4.8 min)
epoch 4  loss 0.4631  val acc 0.795  val macro-F1 0.669  (4.8 min)
{'model': 'esm2_35M_lora+mlp', 'seed': 2, 'val_acc': 0.7948602344454464, 'val_macro_f1': 0.6694226026468993}


val_acc        val_macro_f1       
                     mean    std         mean    std
model                                               
esm2_35M_lora+mlp   0.791  0.004        0.665  0.005